In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import re

# 加载本地模型，并打上LoRA补丁

In [6]:
# 1. 定义两个路径
base_model_path = "./model"         # 你最初下载的、几十亿参数的基础大模型路径
lora_weights_path = "./lora_results" # 你刚才微调保存的 "便利贴" (补丁) 路径

print("1. 正在加载基础模型和分词器 (搬出百科全书)...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.bfloat16, 
    device_map="auto" # 自动调用 Mac 的 mps 加速
)

print("2. 正在融合微调权重 (贴上便利贴)...")
# 【核心魔法】将基础模型和 LoRA 补丁组装在一起
model = PeftModel.from_pretrained(base_model, lora_weights_path)
print("模型融合完毕！\n")

1. 正在加载基础模型和分词器 (搬出百科全书)...


Loading weights: 100%|████████████████████████| 339/339 [00:06<00:00, 53.11it/s]


2. 正在融合微调权重 (贴上便利贴)...
模型融合完毕！



# 开始测试

In [7]:
#初始化全局对话历史列表
messages = []

print("="*50)
print("欢迎测试微调后的专属助手！输入 'quit' or 'exit' 结束。")
print("="*50)

# 3. 开始连续对话测试
while True:
    # 获取用户在终端的输入
    user_input = input("\nUser: ")
    
    # 设置退出指令
    if user_input.lower() in ['quit', 'exit']:  # 将字符串中所有大写字母转换为小写字母
        print("结束对话，再见！")
        break
    if not user_input.strip(): # 去除两端空白字符
        continue

    # 【避坑指南】微调后的推理，必须严格使用训练时的对话格式！
    # 回顾我们刚才在 SFTTrainer 里写的格式：f"User: {instruction}\nAssistant: "
    prompt = f"User: {user_input}\nAssistant: "
    
    # 编码输入
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1]

    # 模型推理生成
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=256,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.3 # 降低温度，让模型回答更严谨
    )

    # 切片截取新生成的 Token
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    # 清理可能存在的 <think> 标签（针对 DeepSeek-R1 等模型）
#     clean_response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    
    print(f"\nAssistant: {response}")

欢迎测试微调后的专属助手！输入 'quit' or 'exit' 结束。

User: 碳中和是什么？

Assistant: 碳中和是指将全球碳排放量从2005年的峰值（约580亿吨）降至2019年的峰值前的水平，即实现碳排放量从580亿吨降至2019年的水平，这相当于将全球碳排放量从峰值前的水平（约580亿吨）降至峰值前的水平（约580亿吨）的水平，这相当于将全球碳排放量从峰值前的水平（约580亿吨）降至峰值前的水平（约580亿吨）的水平，这相当于将全球碳排放量从峰值前的水平（约580亿吨）降至峰值前的水平（约580亿吨）的水平，这相当于将全球碳排放量从峰值前的水平（约580亿吨）降至峰值前的水平（约580亿吨）的水平，这相当于将全球碳排放量从峰值前的水平（约580亿吨）降至峰值前的水平（约580亿吨）的水平，这相当于将全球碳排放量从峰值前的水平（约580亿吨）降至峰值前的水平（约580亿吨）的水平，

User: 什么是微电网？

Assistant: 什么是微电网？微电网是一种新型的智能电网系统，它结合了传统电力系统和智能电网的特点，通过微粒技术实现电力的高效、安全和经济运行。微电网的运行模式基于微粒理论，能够实时调整电力的分配和分配，从而实现电力的优化运行。微电网的智能化和自适应性使其具有较高的灵活性和适应性，能够适应不同地区的电力需求和挑战。微电网的经济性、安全性和高效性使其成为智能电网系统的重要组成部分。

User: 为什么中国的“3060”目标被认为极具挑战性？

Assistant: 中国的“3060”目标，也称为“2030年可持续发展目标”，被认为是极具挑战性的，主要原因在于它要求在2030年前实现碳中和，这意味着需要在2020年前实现碳捕获和利用，这是一项非常大的挑战。此外，中国在能源结构转型方面面临巨大的技术、经济和政策障碍，需要在多个领域进行大规模投资，这使得目标的实现难度极大。此外，中国在应对气候变化方面还面临技术、资金和组织等方面的挑战，需要在多个领域进行大规模投资，这使得目标的实现难度极大。因此，中国在制定“3060”目标时，需要综合考虑这些因素，确保目标的实现具有可行性。

User: quit
结束对话，再见！
